In [133]:
import civicpy.civic as civic
import pandas as pd

import gene_ids_in_lit.functions as fn

In [134]:
evidence = civic.get_all_evidence(include_status="accepted")
molecular_profiles = civic.get_all_molecular_profiles(include_status="accepted")

In [135]:
len(evidence)

4885

In [136]:
# Build lookup by CIViC molecular profile ID
mp_by_id = {
    mp.id: mp
    for mp in molecular_profiles
}

In [137]:
rows = []

for e in evidence:
    mp = mp_by_id.get(e.molecular_profile_id)

    if mp is None:
        continue

    for x in mp.parsed_name:
        if isinstance(x, civic.Gene):
            rows.append({
                "gene_symbol": x.name,
                "entrez_id": x.entrez_id,
                "PMID": e.source.citation_id
            })

In [138]:
df = (
    pd.DataFrame(rows)
    .drop_duplicates()
    .reset_index(drop=True)
)
df

,gene_symbol,entrez_id,PMID
0,NPM1,4869,19357394
1,EGFR,1956,25668228
2,MGMT,4255,15758010
3,KRAS,3845,18946061
4,BRAF,673,21639808
...,...,...,...
2052,RHOA,387,27313181
2053,KRAS,3845,28259530
2054,EGFR,1956,29196463
2055,KIT,3815,28595259


Make merged ambiguous symbol df from alias-alias and alias-primary collisions

In [139]:
aa_colliison_df = pd.read_csv("../output/merged_aa_collision_gene_df.csv")
ap_colliison_df = pd.read_csv("../output/merged_alias_primary_collisions_df.csv")

aa_colliison_df = aa_colliison_df.rename(columns={
    "collision": "ambiguous_symbol",
})

ap_colliison_df = ap_colliison_df.rename(columns={
    "collision": "ambiguous_symbol",
})

aa_colliison_df["collision type"] = "alias-alias"
ap_colliison_df["collision type"] = "alias-primary"

collision_df = pd.concat([aa_colliison_df, ap_colliison_df], ignore_index=True)

key_cols = [
    "ambiguous_symbol",
    "NCBI_ID",
    "primary_gene_symbol"
]

collision_df = (
    collision_df
    .groupby(key_cols, as_index=False)["collision type"]
    .agg(lambda x: sorted(set(x)))
)

In [140]:
collision_df

,ambiguous_symbol,NCBI_ID,primary_gene_symbol,collision type
0,7SK,GENE ID:125050,RN7SK,[alias-primary]
1,A2M,GENE ID:3494,IGHA2,[alias-primary]
2,AAVS1,GENE ID:54776,PPP1R12C,[alias-primary]
3,ACAT1,GENE ID:6646,SOAT1,[alias-primary]
4,ACAT2,GENE ID:8435,SOAT2,[alias-primary]
...,...,...,...,...
1679,ZNF581,GENE ID:29903,CCDC106,[alias-primary]
1680,ZNF688,GENE ID:146540,ZNF785,[alias-primary]
1681,ZP1,GENE ID:57829,ZP4,[alias-primary]
1682,ZSCAN30,GENE ID:55663,ZNF446,[alias-primary]


In [141]:
collision_df = (
    collision_df.groupby(
        ["NCBI_ID", "primary_gene_symbol"],
        as_index=False
    )
    .agg({
        "ambiguous_symbol": lambda x: list(x)
    })
)

In [142]:
collision_df

,NCBI_ID,primary_gene_symbol,ambiguous_symbol
0,GENE ID:100008586,GAGE12F,[GAGE7]
1,GENE ID:100008587,RNA5-8SN5,[RNA5-8S5]
2,GENE ID:100008588,RNA18SN5,[RNA18S5]
3,GENE ID:100008589,RNA28SN5,[RNA28S5]
4,GENE ID:100009602,TRY-GTA5-4,[TRY-GTA5-2]
...,...,...,...
1667,GENE ID:9962,SLC23A2,[SLC23A1]
1668,GENE ID:9963,SLC23A1,[SLC23A2]
1669,GENE ID:9968,MED12,[OPA1]
1670,GENE ID:9988,DMTF1,[DMP1]


Remove genes from civic pmids df that are not involved in collisions

In [143]:
collision_df["entrez_id"] = (
    collision_df["NCBI_ID"]
    .str.extract(r"(\d+)", expand=False)
    .astype("Int64")
)

In [144]:
collision_df

,NCBI_ID,primary_gene_symbol,ambiguous_symbol,entrez_id
0,GENE ID:100008586,GAGE12F,[GAGE7],100008586
1,GENE ID:100008587,RNA5-8SN5,[RNA5-8S5],100008587
2,GENE ID:100008588,RNA18SN5,[RNA18S5],100008588
3,GENE ID:100008589,RNA28SN5,[RNA28S5],100008589
4,GENE ID:100009602,TRY-GTA5-4,[TRY-GTA5-2],100009602
...,...,...,...,...
1667,GENE ID:9962,SLC23A2,[SLC23A1],9962
1668,GENE ID:9963,SLC23A1,[SLC23A2],9963
1669,GENE ID:9968,MED12,[OPA1],9968
1670,GENE ID:9988,DMTF1,[DMP1],9988


make sure types are the same

In [145]:
df["entrez_id"] = df["entrez_id"].astype("Int64")

In [146]:
df_filtered = df[
    df["entrez_id"].isin(collision_df["entrez_id"])
].copy()

In [147]:
df_filtered

,gene_symbol,entrez_id,PMID
11,UGT1A1,54658,26313268
60,NRAS,4893,28275037
89,H3-3A,3020,38335473
95,EZH2,2146,33035457
96,CHEK2,11200,32343890
...,...,...,...
2013,KLF5,688,28963353
2014,PTEN,5728,24088382
2016,FGFR1,2260,34593528
2027,NRAS,4893,24950457


Query pubmed articles for ambiguous symbol

In [148]:
df_filtered = df_filtered.merge(
    collision_df[["entrez_id", "ambiguous_symbol"]],
    on="entrez_id",
    how="left"
)
df_filtered

,gene_symbol,entrez_id,PMID,ambiguous_symbol
0,UGT1A1,54658,26313268,[UGT1A]
1,NRAS,4893,28275037,[KRAS]
2,H3-3A,3020,38335473,[H3-3B]
3,EZH2,2146,33035457,[EZH1]
4,CHEK2,11200,32343890,[CDS1]
...,...,...,...,...
210,KLF5,688,28963353,[CKLF]
211,PTEN,5728,24088382,[TEP1]
212,FGFR1,2260,34593528,[FLG]
213,NRAS,4893,24950457,[KRAS]


In [149]:
import re
import time

In [150]:
def find_aliases_in_document(
    document: dict,
    aliases: list[str],
) -> list[str]:
    """Find ambiguous gene aliases in PubTator document text."""

    found = set()

    for passage in document.get("passages", []):
        text = str(passage.get("text", ""))

        for alias in aliases:
            pattern = re.compile(
                rf"(?<!\w){re.escape(alias)}(?!\w)",
                re.IGNORECASE,
            )

            if pattern.search(text):
                found.add(alias)

    return sorted(found)

In [151]:
pmids = set(
    df_filtered["PMID"]
    .dropna()
    .astype(str)
)

In [152]:
def fetch_documents_by_pmids(
    pmids,
    batch_size=50,
):
    """Fetch PubTator documents for a collection of PMIDs."""

    sorted_pmids = sorted(
        {str(pmid) for pmid in pmids}
    )

    for batch in fn.chunked(sorted_pmids, size=batch_size):

        response = fn.get_with_retry(
            f"{fn.BASE_URL}/publications/export/biocjson",
            params={
                "pmids": ",".join(batch),
                "full": "true",
            },
            timeout=180,
        )

        result = response.json()

        if isinstance(result, list):
            documents = result

        elif isinstance(result, dict) and "PubTator3" in result:
            documents = result["PubTator3"]

        elif isinstance(result, dict) and "documents" in result:
            documents = result["documents"]

        elif isinstance(result, dict) and "id" in result:
            documents = [result]

        else:
            documents = []

        yield from documents

        time.sleep(2)

In [153]:
import importlib

importlib.reload(fn)

<module 'gene_ids_in_lit.functions' from '/Users/rsaxs014/Desktop/gene-harmony-analysis/analysis/gene_ids_in_lit/functions.py'>

In [154]:
pmids = set(
    df_filtered.loc[
        df_filtered["PMID"].astype(str).str.fullmatch(r"\d+"),
        "PMID"
    ]
    .astype(str)
)

In [155]:
documents_by_pmid = {}

for document in fetch_documents_by_pmids(pmids):

    pmid = str(
        document.get("pmid")
        or document.get("id")
        or ""
    ).strip()

    if pmid:
        documents_by_pmid[pmid] = document

In [156]:
def find_row_aliases(row):
    pmid = str(row["PMID"])

    document = documents_by_pmid.get(pmid)

    if document is None:
        return []

    return find_aliases_in_document(
        document,
        row["ambiguous_symbol"],
    )

In [157]:
def document_has_full_text(document):
    passage_types = {
        passage.get("infons", {}).get("type", "unknown")
        for passage in document.get("passages", [])
    }

    return not passage_types <= {"title", "abstract"}

In [158]:
def analyze_row(row):
    pmid = str(row["PMID"])
    document = documents_by_pmid.get(pmid)

    if document is None:
        return pd.Series({
            "full_text_available": False,
            "aliases_found": [],
        })

    return pd.Series({
        "full_text_available": document_has_full_text(document),
        "aliases_found": find_aliases_in_document(
            document,
            row["ambiguous_symbol"],
        ),
    })

In [159]:
results = df_filtered.apply(
    analyze_row,
    axis=1,
)

df_filtered = pd.concat(
    [df_filtered, results],
    axis=1,
)

In [160]:
df_filtered = df_filtered.reset_index(drop=True)

df_filtered = df_filtered[
    df_filtered["aliases_found"].apply(len) > 0
].reset_index(drop=True)

In [161]:
df_filtered

,gene_symbol,entrez_id,PMID,ambiguous_symbol,full_text_available,aliases_found
0,NRAS,4893,28275037,[KRAS],False,[KRAS]
1,NRAS,4893,25666295,[KRAS],False,[KRAS]
2,CHEK2,11200,10617473,[CDS1],False,[CDS1]
3,EZH2,2146,20081860,[EZH1],True,[EZH1]
4,PTEN,5728,17700571,[TEP1],True,[TEP1]
5,NRG1,3084,26137564,[HRG],True,[HRG]
6,NRAS,4893,24666267,[KRAS],False,[KRAS]
7,NRAS,4893,15951308,[KRAS],False,[KRAS]
8,NRAS,4893,20619739,[KRAS],False,[KRAS]
9,NRAS,4893,22650231,[KRAS],False,[KRAS]


dgidb

In [162]:
dgidb_genes_df = pd.read_csv("../input/gene_claims (2).csv")

In [163]:
ambiguous_dgidb_genes_df = dgidb_genes_df[
    dgidb_genes_df["name"].isin(collision_df["ambiguous_symbol"].explode().dropna())
].copy()

In [164]:
# Split pipe-delimited values into individual entries
items = (
    ambiguous_dgidb_genes_df["aliases"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
)

# Keep entries containing ":" and extract everything before the first ":"
prefixes = (
    items[items.str.contains(":", regex=False)]
    .str.split(":", n=1)
    .str[0]
)

In [165]:
prefix_counts = prefixes.value_counts()

In [166]:
unprefixed = items[~items.str.contains(":", regex=False)]

print(unprefixed.head(50))

23                                                 DEFA1
28                                                    HD
28                                            HUNTINGTIN
28                                                  IT15
42                                                  CCR4
42                                            CCR4_HUMAN
49                                   interferon alpha 13
86                                                   CNP
109                                                 MEKA
109                                                  PHD
109                                                PhLOP
109                                                 PhLP
109                                            phosducin
124                                            LOC730082
124                                            LOC731373
124                                          neuroglobin
162                                    LIM domain only 4
211                            

In [167]:
prefixed = items[items.str.contains(":", regex=False)].to_frame("value")

prefixed["prefix"] = prefixed["value"].str.split(":", n=1).str[0]

summary = (
    prefixed.groupby("prefix")
    .agg(
        count=("value", "size"),
        example=("value", "first")
    )
    .sort_values("count", ascending=False)
)

print(summary)

                 count                                            example
prefix                                                                   
hgnc              2261                                          hgnc:5419
ensembl           2240                            ensembl:ENSG00000233816
ncbigene          1519                                      ncbigene:5132
omim              1447                                        omim:171490
ccds              1444                                     ccds:CCDS47626
pubmed            1162                                     pubmed:1837206
UNIPROT            893                                     UNIPROT:P51679
refseq             760                                   refseq:NM_000601
uniprot            751                                     uniprot:P14210
vega               745                            vega:OTTHUMG00000023804
ucsc               733                                    ucsc:uc003uhl.4
ena.embl           621                

In [168]:
ambiguous_dgidb_genes_df

,name,nomenclature,source_db_name,normalized_gene_id,aliases
23,DEFA1,Gene Name,HingoraniCasas,hgnc:2761,DEFA1|ENSEMBL:ENSG00000206047
28,HTT,Gene Symbol,GO,hgnc:4851,HD|HUNTINGTIN|IT15
42,CCR4,Gene Name,DrugBank,hgnc:1605,CCR4|CCR4_HUMAN|HGNC:1605|IUPHAR.RECEPTOR:61|U...
49,IFNA13,Gene Symbol,Ensembl,hgnc:5419,ensembl:ENSG00000233816|hgnc:5419|interferon a...
86,CNP,Gene Name,HingoraniCasas,hgnc:2158,CNP|ENSEMBL:ENSG00000173786
...,...,...,...,...,...
79542,DUSP1,Gene Name,NCI,hgnc:3064,NaN
79610,ART1,Gene Name,DrugBank,hgnc:723,ART1|GENBANK:807100|HGNC:723|NAR1_HUMAN|S74683...
79717,MCM2,Gene Symbol,HGNC,hgnc:6944,BM28|ccds:CCDS3043|CCNL1|cdc19|CDCL1|D3S3194|D...
79729,SPP1,Gene Name,PharmGKB,hgnc:11255,PHARMGKB.GENE:PA36085


In [169]:
prefixes = ["hgnc", "ensembl", "ncbigene", "ncbi.gene"]

pattern = r"(?:^|\|)(?:" + "|".join(map(re.escape, prefixes)) + r"):"

mask = ambiguous_dgidb_genes_df["aliases"].str.contains(
    pattern,
    case=False,
    regex=True,
    na=False
)

# Number of rows that have any of those prefixes
ambiguous_dgidb_genes_with_identifiers_df = ambiguous_dgidb_genes_df[mask]

In [170]:
import re

ambiguous_dgidb_genes_with_identifiers_df["HGNC_ID"] = ambiguous_dgidb_genes_with_identifiers_df["aliases"].str.extract(
    r"(?:^|\|)(?:hgnc|HGNC):([^|]+)",
    expand=False
)

ambiguous_dgidb_genes_with_identifiers_df["NCBI_ID"] = ambiguous_dgidb_genes_with_identifiers_df["aliases"].str.extract(
    r"(?:^|\|)(?:ncbigene|NCBIGENE|NCBI\.GENE):([^|]+)",
    expand=False
)

ambiguous_dgidb_genes_with_identifiers_df["ENSG_ID"] = ambiguous_dgidb_genes_with_identifiers_df["aliases"].str.extract(
    r"(?:^|\|)(?:ensembl|ENSEMBL):(ENSG[^|]+)",
    expand=False
)

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_42303/2097094404.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_dgidb_genes_with_identifiers_df["HGNC_ID"] = ambiguous_dgidb_genes_with_identifiers_df["aliases"].str.extract(
/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_42303/2097094404.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ambiguous_dgidb_genes_with_identifiers_df["NCBI_ID"] = ambiguous_dgidb_genes_with_identifiers_df["aliases"].str.extract(
/var/folders/v

In [171]:
ambiguous_dgidb_genes_with_identifiers_df = ambiguous_dgidb_genes_with_identifiers_df.drop(columns=["nomenclature"])

In [172]:
capture_df = pd.read_csv("../output/summary_df.csv")

In [173]:
capture_df = capture_df[capture_df["captured"]=="T"]
capture_df

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:
0,0,set(),set(),{'GENE ID:109951028'},A-GAMMA3'E,A-GAMMA3'E,T,Primary Gene Symbol
2,2,{'HGNC:5'},{'ENSG00000121410'},{'GENE ID:1'},A1BG,A1B,T,Alternate Abbreviation Symbol
3,3,{'HGNC:5'},{'ENSG00000121410'},{'GENE ID:1'},A1BG,A1BG,T,Primary Gene Symbol
7,7,{'HGNC:37133'},{'ENSG00000268895'},{'GENE ID:503538'},A1BG-AS1,A1BG-AS,T,Previous Symbol
8,8,{'HGNC:37133'},{'ENSG00000268895'},{'GENE ID:503538'},A1BG-AS1,A1BG-AS1,T,Primary Gene Symbol
...,...,...,...,...,...,...,...,...
128221,128221,{'HGNC:29027'},{'ENSG00000074755'},{'GENE ID:23140'},ZZEF1,ZZEF1,T,Primary Gene Symbol
128222,128222,{'HGNC:29027'},{'ENSG00000074755'},{'GENE ID:23140'},ZZEF1,ZZZ4,T,Prefix Gene Group Symbol
128223,128223,{'HGNC:24523'},{'ENSG00000036549'},{'GENE ID:26009'},ZZZ3,ATAC1,T,"Ortholog Symbol, Prefix Gene Group Symbol"
128224,128224,{'HGNC:24523'},{'ENSG00000036549'},{'GENE ID:26009'},ZZZ3,DKFZP564I052,T,Placeholder Symbol


In [174]:
ambiguous_symbols = set(capture_df["gene_symbol"])
dgidb_ambiguous_symbols = set(ambiguous_dgidb_genes_with_identifiers_df["name"])


In [176]:
len(ambiguous_symbols & dgidb_ambiguous_symbols)

733

In [ ]:
category_list = ['Primary Gene Symbol',
                'Previous Symbol',
                'Clone Name Symbol',
                'Prefix Gene Group Symbol',
                "Ortholog Symbol", 
                "Prefix Condition Symbol",
                "Protein Mass Symbol", 
                "Placeholder Symbol", 
                "Gene Identifier Symbol",
                "Gene Neighbor Symbol",
                "Gene Group Symbol", 
                "Gene Interaction Symbol",
                "Withdrawn Ortholog Symbol"]